# Tractor Calculus: The Conformal Toolkit's Algebraic Backbone

This notebook introduces the **tractor calculus** module of `conformal_toolkit`.
Tractor calculus is the conformal geometer's analogue of the spinor calculus
familiar from general relativity: it packages conformal data into a single
vector bundle equipped with a canonical connection, making conformal invariance
*manifest* rather than something to be checked after the fact.

## What is a tractor?

On an $n$-dimensional conformal manifold $(M, [g])$, the **standard tractor bundle**
is a rank-$(n+2)$ vector bundle $\mathcal{T}$ carrying a natural $O(p+1, q+1)$
inner product and a canonical connection $\nabla^{\mathcal{T}}$. Once a metric
representative $g \in [g]$ is chosen (a "conformal scale"), a section of
$\mathcal{T}$ splits into a triple:
$$I^A = \begin{pmatrix} \sigma \\ \mu_a \\ \rho \end{pmatrix}$$
where $\sigma$ is a scalar (weight $+1$ density), $\mu_a$ is a 1-form
(weight $-1$), and $\rho$ is a scalar (weight $-1$).

The **tractor metric** (inner product) is:
$$h(I, J) = \sigma \rho' + \rho \sigma' + g^{ab}\mu_a \mu'_b$$

The **normal tractor connection** acts as:
$$\nabla^{\mathcal{T}}_a \begin{pmatrix} \sigma \\ \mu_b \\ \rho \end{pmatrix}
= \begin{pmatrix} \nabla_a \sigma - \mu_a \\
\nabla_a \mu_b + P_{ab}\sigma + g_{ab}\rho \\
\nabla_a \rho - P_a{}^b \mu_b \end{pmatrix}$$

The curvature of this connection is the **tractor curvature**, whose key component
is the Weyl tensor $W^a{}_{bcd}$. On conformally flat manifolds ($S^n$, $\mathbb{R}^n$),
the tractor curvature vanishes.

The **Thomas $D$-operator** maps conformal densities of weight $w$ to tractors:
$$D_A f = \begin{pmatrix} (n+2w-2)w\,f \\ (n+2w-2)\nabla_a f \\ -\Delta f - wJf \end{pmatrix}$$
where $J = \mathrm{tr}(P)$ is the Schouten trace.

## 1. Creating a standard tractor on flat $\mathbb{R}^3$

On flat space, $P = 0$ and $J = 0$, so the tractor connection reduces to
ordinary derivatives (up to the slot-mixing terms involving $\mu$).
We create a tractor with $\sigma = x_0$, $\mu = dx_0$, $\rho = 1$ and
compute its tractor inner product and connection.

In [ ]:
from sage.all import Manifold
from conformal_toolkit.core.conformal_structure import ConformalStructure
from conformal_toolkit.tractor.standard_tractor import StandardTractor
from conformal_toolkit.tractor.tractor_metric import tractor_inner
from conformal_toolkit.tractor.tractor_connection import tractor_connection
from conformal_toolkit.tractor.tractor_curvature import tractor_curvature, cotton_tensor
from conformal_toolkit.tractor.thomas_d import thomas_d

# Flat R^3
R3 = Manifold(3, 'R3', structure='Riemannian')
X = R3.chart('x0 x1 x2')
x0, x1, x2 = X[:]
g = R3.metric('g')
for i in range(3):
    g[i, i] = 1

cs = ConformalStructure(g)
print("Flat R^3 metric:")
g.display()

In [ ]:
# Create a tractor section: I = (sigma, mu, rho) = (x0, dx0, 1)
sigma = R3.scalar_field(x0, name='sigma')

# mu is a 1-form: dx0
mu = R3.one_form(name='mu')
mu[0] = 1  # dx0 component
mu[1] = 0
mu[2] = 0

rho = R3.scalar_field(1, name='rho')

I = StandardTractor(cs, sigma, mu, rho)
print("Standard tractor I:")
print(f"  sigma = {I.sigma.display()}")
print(f"  mu = {I.mu.display()}")
print(f"  rho = {I.rho.display()}")

In [ ]:
# Tractor inner product h(I, I)
# h(I, I) = sigma*rho + rho*sigma + g^{ab} mu_a mu_b
#         = x0*1 + 1*x0 + 1*1 + 0 + 0
#         = 2*x0 + 1
h_II = tractor_inner(cs, I, I)
print("Tractor inner product h(I, I):")
h_II.display()

# Output:
# h(I, I) = 2*x0 + 1

In [ ]:
# Create a second tractor J = (1, 0, x0) and compute h(I, J)
sigma2 = R3.scalar_field(1, name='sigma2')
mu2 = R3.one_form(name='mu2')
mu2[0] = 0
mu2[1] = 0
mu2[2] = 0
rho2 = R3.scalar_field(x0, name='rho2')

J_tractor = StandardTractor(cs, sigma2, mu2, rho2)

h_IJ = tractor_inner(cs, I, J_tractor)
print("Tractor inner product h(I, J) where J = (1, 0, x0):")
h_IJ.display()

# h(I, J) = sigma*rho' + rho*sigma' + g^{ab} mu_a mu'_b
#         = x0*x0 + 1*1 + 0
#         = x0^2 + 1

## 2. The tractor connection on flat $\mathbb{R}^3$

Applying the tractor connection to $I = (x_0, dx_0, 1)$ on flat space
(where $P = 0$):
$$\nabla^{\mathcal{T}}_a I = \begin{pmatrix}
\nabla_a x_0 - (dx_0)_a \\
\nabla_a (dx_0)_b + 0 + g_{ab} \cdot 1 \\
\nabla_a(1) - 0
\end{pmatrix}
= \begin{pmatrix}
\delta_{a0} - \delta_{a0} \\
0 + g_{ab} \\
0
\end{pmatrix}
= \begin{pmatrix} 0 \\ g_{ab} \\ 0 \end{pmatrix}$$

In [ ]:
# Apply the tractor connection
nabla_T = tractor_connection(cs, I)

print("Tractor connection applied to I = (x0, dx0, 1):")
print("\n  Sigma slot (should be 0):")
nabla_T['sigma'].display()

print("\n  Mu slot (should be g_{ab}):")
nabla_T['mu'].display()

print("\n  Rho slot (should be 0):")
nabla_T['rho'].display()

# Output:
# Sigma slot: 0 (nabla_a(x0) - (dx0)_a = delta_{a0} - delta_{a0} = 0)
# Mu slot: g_{ab} = dx0*dx0 + dx1*dx1 + dx2*dx2
# Rho slot: 0 (nabla_a(1) = 0, P = 0)

## 3. Tractor curvature on $S^2$

The tractor curvature's key component is the Weyl tensor. In dimension $\le 3$,
the Weyl tensor vanishes identically (the Cotton tensor is the obstruction to
conformal flatness in 3D). The round $S^2$ is conformally flat, so both
the tractor curvature and the Cotton tensor should vanish.

In [ ]:
from sage.all import Manifold, sin, cos

# Round S^2
S = Manifold(2, 'S2', structure='Riemannian')
Y = S.chart(r'theta:(0,pi):\theta phi:(0,2*pi):\phi')
theta, phi = Y[:]
g_s2 = S.metric('g')
g_s2[0, 0] = 1
g_s2[1, 1] = sin(theta)**2

cs_s2 = ConformalStructure(g_s2)

# Tractor curvature (Weyl tensor component)
W = tractor_curvature(cs_s2)
print("Tractor curvature (Weyl tensor) on S^2:")
W.display()

# Output:
# W = 0  (identically zero in dimension 2)

In [ ]:
# Cotton tensor on flat R^3 (should vanish: conformally flat)
C = cotton_tensor(cs)
print("Cotton tensor on flat R^3:")
C.display()

# Output:
# C = 0  (flat space is conformally flat)

## 4. The Thomas $D$-operator

The Thomas $D$-operator maps conformal densities to standard tractors. It is
the universal machine for building conformally invariant differential operators:
the GJMS operators can all be constructed from compositions of $D$ with the
tractor inner product.

$$D_A f = \begin{pmatrix}
(n+2w-2)w\,f \\
(n+2w-2)\nabla_a f \\
-\Delta f - wJf
\end{pmatrix}$$

We apply $D$ to the coordinate function $x_0$ on flat $\mathbb{R}^3$, treated
as a density of weight $w = 1$.

In [ ]:
# Thomas D on flat R^3: f = x0, weight w = 1
# n = 3, w = 1
# coeff = n + 2w - 2 = 3 + 2 - 2 = 3
# sigma = 3 * 1 * x0 = 3*x0
# mu = 3 * nabla(x0) = 3 * dx0
# rho = -Delta(x0) - 1 * J * x0 = 0 - 0 = 0

f = R3.scalar_field(x0, name='f')
D_f = thomas_d(cs, f, weight=1)

print("Thomas D-operator applied to f = x0 (weight 1) on flat R^3:")
print(f"  sigma slot: {D_f.sigma.display()}")
print(f"  mu slot: {D_f.mu.display()}")
print(f"  rho slot: {D_f.rho.display()}")

# Output:
# sigma slot: 3*x0
# mu slot: 3*dx0
# rho slot: 0

In [ ]:
# Thomas D at weight 0: D_A(1) with f = 1, w = 0
# coeff = n + 2*0 - 2 = n - 2 = 1
# sigma = 1 * 0 * 1 = 0
# mu = 1 * nabla(1) = 0
# rho = -Delta(1) - 0*J*1 = 0

f_const = R3.scalar_field(1, name='one')
D_one = thomas_d(cs, f_const, weight=0)

print("Thomas D(1) at weight 0 on flat R^3:")
print(f"  sigma = {D_one.sigma.display()}")
print(f"  mu = {D_one.mu.display()}")
print(f"  rho = {D_one.rho.display()}")

# Output: all zero -- D annihilates weight-0 constants.

In [ ]:
# Thomas D applied to f = x0^2 + x1^2 at weight 1
# n = 3, w = 1, coeff = 3
# sigma = 3 * 1 * (x0^2 + x1^2) = 3(x0^2 + x1^2)
# mu = 3 * nabla(x0^2 + x1^2) = 3 * (2*x0*dx0 + 2*x1*dx1)
#    = 6*x0 dx0 + 6*x1 dx1
# rho = -Delta(x0^2 + x1^2) - 1*0*(x0^2+x1^2) = -4

f_quad = R3.scalar_field(x0**2 + x1**2, name='f_quad')
D_quad = thomas_d(cs, f_quad, weight=1)

print("Thomas D(x0^2 + x1^2) at weight 1 on flat R^3:")
print(f"  sigma = {D_quad.sigma.display()}")
print(f"  mu = {D_quad.mu.display()}")
print(f"  rho = {D_quad.rho.display()}")

# Output:
# sigma = 3*(x0^2 + x1^2)
# mu = 6*x0 dx0 + 6*x1 dx1
# rho = -4
#
# The rho slot captures the Laplacian of f, which is the key
# link between the Thomas D-operator and the GJMS operators.

## Summary

This notebook demonstrated the tractor calculus module of `conformal_toolkit`:

1. **Standard tractors** package conformal data into $(\sigma, \mu_a, \rho)$ triples
2. **Tractor inner product** $h(I, J) = \sigma\rho' + \rho\sigma' + g^{ab}\mu_a\mu'_b$
   is the natural $O(n+1, 1)$ pairing
3. **Tractor connection** reduces to ordinary derivatives on flat space, with
   corrections from the Schouten tensor on curved spaces
4. **Tractor curvature** is the Weyl tensor (vanishes on conformally flat spaces)
5. **Cotton tensor** is the 3D obstruction to conformal flatness
6. **Thomas $D$-operator** maps densities to tractors and encodes the Laplacian
   in its bottom slot

### Why tractors matter

Tractors provide a *systematic* way to construct conformally invariant operators
and quantities. Rather than guessing correction terms and checking conformal
covariance by hand, one builds conformally invariant objects by composing
tractor operations. This is how the GJMS operators, $Q$-curvatures, and many
other invariants were originally discovered.

**Next:** Notebook 04 applies these ideas to the holographic setting via
Poincare-Einstein metrics.